# MCSDCA PushT Predictor Results

In [11]:
from __future__ import annotations

import csv
import json
from pathlib import Path

try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass

    def display(value):
        print(value)

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src" / "run_pusht_predictor_experiment.py").exists():
        ROOT = candidate
        break
else:
    raise RuntimeError("Cannot find repo root.")

# Set this to a specific run folder if you do not want the latest run.
RUN_DIR = None
OUTPUT_ROOT = ROOT / "outputs" / "pusht_predictor_optimizer"


def latest_run(root: Path) -> Path:
    runs = [path for path in root.iterdir() if path.is_dir() and (path / "summary.json").exists()]
    if not runs:
        raise FileNotFoundError(f"No experiment runs found under {root}. Run src/run_pusht_predictor_experiment.py first.")
    return max(runs, key=lambda path: path.stat().st_mtime)


run_dir = Path(RUN_DIR).resolve() if RUN_DIR else latest_run(OUTPUT_ROOT)
display(Markdown(f"Using run: `{run_dir}`"))

Using run: `D:\DCA\MCSDCA\outputs\pusht_predictor_optimizer\20260810_031900`

In [12]:
def read_json(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def read_csv(path: Path):
    with path.open("r", encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


def format_value(value):
    if value is None:
        return ""
    text = str(value)
    try:
        number = float(text)
    except ValueError:
        return text
    if number == 0:
        return "0"
    if abs(number) >= 1000 or abs(number) < 1e-3:
        return f"{number:.3e}"
    return f"{number:.6f}"


def markdown_table(rows):
    if not rows:
        return "No rows."
    headers = list(rows[0].keys())
    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for row in rows:
        lines.append("| " + " | ".join(format_value(row.get(header, "")) for header in headers) + " |")
    return "\n".join(lines)


def show_table(title, rows):
    display(Markdown(f"### {title}\n\n" + markdown_table(rows)))


summary = read_json(run_dir / "summary.json")
config = read_json(run_dir / "config.json")
metrics = read_csv(run_dir / "metrics_step.csv")
predictor_table = read_csv(run_dir / "predictor_table.csv")
rollout_table = read_csv(run_dir / "rollout_table.csv")
planning_table = read_csv(run_dir / "planning_table.csv")

In [13]:
args = config["args"]
dataset = config["dataset"]
config_rows = [
    {"Key": "data_path", "Value": args["data_path"]},
    {"Key": "checkpoint_dir", "Value": args["checkpoint_dir"]},
    {"Key": "optimizers", "Value": args["optimizers"]},
    {"Key": "device", "Value": config["device"]},
    {"Key": "history_size", "Value": args["history_size"]},
    {"Key": "num_preds", "Value": args["num_preds"]},
    {"Key": "frameskip", "Value": args["frameskip"]},
    {"Key": "batch_size", "Value": args["batch_size"]},
    {"Key": "train_batches", "Value": args["train_batches"]},
    {"Key": "val_batches", "Value": args["val_batches"]},
    {"Key": "backprop_budget", "Value": args["backprop_budget"]},
    {"Key": "dataset pixel_shape", "Value": dataset["pixel_shape"]},
    {"Key": "dataset action_shape", "Value": dataset["action_shape"]},
]
show_table("Run config", config_rows)

### Run config

| Key | Value |
| --- | --- |
| data_path | D:\DCA\MCSDCA\data\pusht_expert_train.h5 |
| checkpoint_dir | D:\DCA\MCSDCA\data\checkpoints\pusht\lewm |
| optimizers | AdamW,MCSDCA-odLD,MCSDCA-udLD |
| device | cpu |
| history_size | 3.000000 |
| num_preds | 1.000000 |
| frameskip | 5.000000 |
| batch_size | 2.000000 |
| train_batches | 20.000000 |
| val_batches | 5.000000 |
| backprop_budget | 100.000000 |
| dataset pixel_shape | [2336736, 224, 224, 3] |
| dataset action_shape | [2336736, 2] |

In [14]:
show_table("6.1 One-step predictor training", predictor_table)

### 6.1 One-step predictor training

| Optimizer | Backprop calls | Train MSE | Val MSE | Train/Val gap | Latent drift | Time |
| --- | --- | --- | --- | --- | --- | --- |
| AdamW | 100.000000 | 0.228738 | 0.209874 | -0.018864 | 8.982445 | 6.132528 |
| MCSDCA-odLD | 100.000000 | 0.606596 | 0.607901 | 0.001305 | 14.276592 | 19.088203 |
| MCSDCA-udLD | 100.000000 | 0.573904 | 0.530366 | -0.043538 | 13.290040 | 29.380859 |

In [15]:
show_table("6.2 Multi-step rollout loss", rollout_table)

### 6.2 Multi-step rollout loss

| Optimizer | Horizon | Rollout MSE@1 | Rollout MSE@3 | Rollout MSE@5 | Latent drift | Time |
| --- | --- | --- | --- | --- | --- | --- |
| AdamW | 5.000000 | 0.217135 | 0.622057 | 1.079101 | 8.982445 | 6.132528 |
| MCSDCA-odLD | 5.000000 | 0.643644 | 2.311535 | 3.768106 | 14.276592 | 19.088203 |
| MCSDCA-udLD | 5.000000 | 0.561033 | 1.954506 | 3.290971 | 13.290040 | 29.380859 |

In [16]:
show_table("6.3 PushT planning", planning_table)

### 6.3 PushT planning

| Optimizer | CEM horizon | Eval episodes | PushT score | CEM cost | Eval time |
| --- | --- | --- | --- | --- | --- |
| AdamW | TBD | TBD | TBD | TBD | Skipped |
| MCSDCA-odLD | TBD | TBD | TBD | TBD | Skipped |
| MCSDCA-udLD | TBD | TBD | TBD | TBD | Skipped |

In [17]:
metric_preview = metrics[-20:] if len(metrics) > 20 else metrics
show_table("Step metrics preview", metric_preview)

### Step metrics preview

| backprop_calls | gamma_k | latent_norm_drift | markov_chain_length | optimizer | outer_step | pred_latent_variance | retained_samples | rollout_mse_1 | rollout_mse_3 | rollout_mse_5 | sampler_loss | time_s | train_mse | train_val_gap | val_mse |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 93.000000 | 1.328e-05 | 14.089734 | 8.000000 | MCSDCA-odLD | 17.000000 | 1.657893 | 7.000000 | 0.605392 | 1.969045 | 3.317823 | 0.620532 | 17.890405 | 0.526484 | 0.052990 | 0.579474 |
| 100.000000 | 1.335e-05 | 14.276592 | 8.000000 | MCSDCA-odLD | 18.000000 | 1.604540 | 7.000000 | 0.643644 | 2.311535 | 3.768106 | 0.600860 | 19.088203 | 0.606596 | 0.001305 | 0.607901 |
| 4.000000 | 1.000e-05 | 2.621528 | 5.000000 | MCSDCA-udLD | 1.000000 | 0.826321 | 4.000000 | 0.064199 | 0.183563 | 0.345463 | 0.639231 | 0.840203 | 0.062316 | 0.001483 | 0.063799 |
| 8.000000 | 1.072e-05 | 4.318524 | 5.000000 | MCSDCA-udLD | 2.000000 | 0.942058 | 4.000000 | 0.122060 | 0.376882 | 0.728289 | 0.574836 | 2.203592 | 0.113976 | 0.001343 | 0.115320 |
| 12.000000 | 1.116e-05 | 7.061863 | 5.000000 | MCSDCA-udLD | 3.000000 | 1.230732 | 4.000000 | 0.248089 | 0.737747 | 1.299892 | 0.744921 | 3.485691 | 0.229119 | 0.001336 | 0.230455 |
| 17.000000 | 1.149e-05 | 7.699116 | 6.000000 | MCSDCA-udLD | 4.000000 | 1.281289 | 5.000000 | 0.261541 | 0.680384 | 1.149831 | 0.640799 | 4.942735 | 0.232308 | 0.008542 | 0.240850 |
| 22.000000 | 1.175e-05 | 7.870476 | 6.000000 | MCSDCA-udLD | 5.000000 | 1.274188 | 5.000000 | 0.293730 | 0.791489 | 1.323735 | 0.623896 | 6.360518 | 0.272342 | 0.007175 | 0.279517 |
| 27.000000 | 1.196e-05 | 7.775762 | 6.000000 | MCSDCA-udLD | 6.000000 | 1.311807 | 5.000000 | 0.252281 | 0.620954 | 0.989350 | 0.603899 | 7.768228 | 0.254121 | -0.009671 | 0.244450 |
| 32.000000 | 1.215e-05 | 9.518283 | 6.000000 | MCSDCA-udLD | 7.000000 | 1.580287 | 5.000000 | 0.342997 | 0.867974 | 1.384546 | 0.650383 | 9.322576 | 0.339198 | -0.002095 | 0.337103 |
| 37.000000 | 1.231e-05 | 11.215638 | 6.000000 | MCSDCA-udLD | 8.000000 | 1.624086 | 5.000000 | 0.452531 | 1.185752 | 1.861752 | 0.660500 | 10.804799 | 0.403163 | 0.048096 | 0.451258 |
| 43.000000 | 1.246e-05 | 9.266732 | 7.000000 | MCSDCA-udLD | 9.000000 | 1.423620 | 6.000000 | 0.369976 | 0.940792 | 1.409543 | 0.725208 | 12.719868 | 0.353530 | 0.007460 | 0.360990 |
| 49.000000 | 1.259e-05 | 11.593353 | 7.000000 | MCSDCA-udLD | 10.000000 | 1.547805 | 6.000000 | 0.540253 | 1.403426 | 2.043593 | 0.609247 | 14.480762 | 0.455146 | 0.066196 | 0.521342 |
| 55.000000 | 1.271e-05 | 13.698636 | 7.000000 | MCSDCA-udLD | 11.000000 | 1.747994 | 6.000000 | 0.681365 | 1.991962 | 2.992502 | 0.608432 | 16.278764 | 0.532983 | 0.135038 | 0.668021 |
| 61.000000 | 1.282e-05 | 12.951565 | 7.000000 | MCSDCA-udLD | 12.000000 | 1.668998 | 6.000000 | 0.705974 | 1.854356 | 2.809528 | 0.554789 | 18.011774 | 0.580085 | 0.132563 | 0.712647 |
| 67.000000 | 1.292e-05 | 9.854048 | 7.000000 | MCSDCA-udLD | 13.000000 | 1.265018 | 6.000000 | 0.472661 | 1.133021 | 1.828683 | 0.609084 | 19.758735 | 0.401397 | 0.058629 | 0.460026 |
| 73.000000 | 1.302e-05 | 13.956715 | 7.000000 | MCSDCA-udLD | 14.000000 | 1.718836 | 6.000000 | 0.681018 | 1.836906 | 2.907662 | 0.707297 | 21.576292 | 0.580296 | 0.063080 | 0.643375 |
| 79.000000 | 1.311e-05 | 14.332667 | 7.000000 | MCSDCA-udLD | 15.000000 | 1.869806 | 6.000000 | 0.620301 | 1.818594 | 3.007901 | 0.591285 | 23.292169 | 0.562585 | 0.028058 | 0.590643 |
| 86.000000 | 1.320e-05 | 11.891503 | 8.000000 | MCSDCA-udLD | 16.000000 | 1.600204 | 7.000000 | 0.514868 | 1.359149 | 2.153874 | 0.588048 | 25.296564 | 0.427048 | 0.067652 | 0.494700 |
| 93.000000 | 1.328e-05 | 12.960342 | 8.000000 | MCSDCA-udLD | 17.000000 | 1.723470 | 7.000000 | 0.526287 | 1.636671 | 2.797918 | 0.539800 | 27.376658 | 0.493537 | 0.013503 | 0.507039 |
| 100.000000 | 1.335e-05 | 13.290040 | 8.000000 | MCSDCA-udLD | 18.000000 | 1.695243 | 7.000000 | 0.561033 | 1.954506 | 3.290971 | 0.504142 | 29.380859 | 0.573904 | -0.043538 | 0.530366 |